In [0]:
%run "../../commons/commons_imports"

In [0]:
df_meta_uf = read(
    base_path=BRONZE_PATH,
    table_name=METAS_UF,
    recursive_by_year=False,
    format = "delta",
)

df_meta_br = read(
    base_path=BRONZE_PATH,
    table_name=METAS_BR,
    recursive_by_year=False,
    format = "delta",
)

df_meta_municipio = read(
    base_path=BRONZE_PATH,
    table_name=METAS_MUNICIPIO,
    recursive_by_year=False,
    format = "delta",
)

In [0]:
df_meta_uf_gold = (
    df_meta_uf
    # =====================================================
    # Padronização
    # =====================================================
    .withColumn(
        "REDE",
        initcap(trim(col("REDE")))
    )
    # =====================================================
    # Código da UF
    # =====================================================
    .withColumn(
        "CO_UF",
        when(col("SG_UF") == "RO", 11)
        .when(col("SG_UF") == "AC", 12)
        .when(col("SG_UF") == "AM", 13)
        .when(col("SG_UF") == "RR", 14)
        .when(col("SG_UF") == "PA", 15)
        .when(col("SG_UF") == "AP", 16)
        .when(col("SG_UF") == "TO", 17)
        .when(col("SG_UF") == "MA", 21)
        .when(col("SG_UF") == "PI", 22)
        .when(col("SG_UF") == "CE", 23)
        .when(col("SG_UF") == "RN", 24)
        .when(col("SG_UF") == "PB", 25)
        .when(col("SG_UF") == "PE", 26)
        .when(col("SG_UF") == "AL", 27)
        .when(col("SG_UF") == "SE", 28)
        .when(col("SG_UF") == "BA", 29)
        .when(col("SG_UF") == "MG", 31)
        .when(col("SG_UF") == "ES", 32)
        .when(col("SG_UF") == "RJ", 33)
        .when(col("SG_UF") == "SP", 35)
        .when(col("SG_UF") == "PR", 41)
        .when(col("SG_UF") == "SC", 42)
        .when(col("SG_UF") == "RS", 43)
        .when(col("SG_UF") == "MS", 50)
        .when(col("SG_UF") == "MT", 51)
        .when(col("SG_UF") == "GO", 52)
        .when(col("SG_UF") == "DF", 53)
    )
    # =====================================================
    # Região
    # =====================================================
    .withColumn(
        "REGIAO",
        when(col("SG_UF").isin("AC","AP","AM","PA","RO","RR","TO"), "Norte")
        .when(col("SG_UF").isin("AL","BA","CE","MA","PB","PE","PI","RN","SE"), "Nordeste")
        .when(col("SG_UF").isin("DF","GO","MS","MT"), "Centro-Oeste")
        .when(col("SG_UF").isin("ES","MG","RJ","SP"), "Sudeste")
        .when(col("SG_UF").isin("PR","RS","SC"), "Sul")
    )
    # =====================================================
    # Quantidade de metas disponíveis
    # =====================================================
    .withColumn(
        "QTD_METAS_DEFINIDAS",
        (
            col("META_ALFABETIZACAO_2024").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2025").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2026").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2027").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2028").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2029").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2030").isNotNull().cast("int")
        )
    )
    # =====================================================
    # Possui metas cadastradas
    # =====================================================
    .withColumn(
        "IN_POSSUI_META",
        when(col("QTD_METAS_DEFINIDAS") > 0, 1)
        .otherwise(0)
    )
)

In [0]:
df_meta_estado_gold_selected = (
    df_meta_uf_gold
    .select(
        "SK_META_ESTADO",
        col("ANO").alias('ANO_REFERENCIA'),
        "CO_UF",
        "SG_UF",
        "REGIAO",
        "REDE",
        "TAXA_ALFABETIZACAO",
        "META_ALFABETIZACAO_2024",
        "META_ALFABETIZACAO_2025",
        "META_ALFABETIZACAO_2026",
        "META_ALFABETIZACAO_2027",
        "META_ALFABETIZACAO_2028",
        "META_ALFABETIZACAO_2029",
        "META_ALFABETIZACAO_2030",
        "QTD_METAS_DEFINIDAS",
        "IN_POSSUI_META",
        "PERCENTUAL_PARTICIPACAO",
        "DT_PROCESSAMENTO",
        "TS_PROCESSAMENTO"
    )

)

In [0]:
validate_primary_key(df_meta_estado_gold_selected, "SK_META_ESTADO")

validate_not_null(
    df_meta_estado_gold_selected,
    [
        "SK_META_ESTADO"
    ]
)

validate_years(df_meta_estado_gold_selected)

In [0]:
write_delta(
    df=df_meta_estado_gold_selected,
    base_path=SILVER_PATH,
    table_name=METAS_UF,
    merge_keys=["SK_META_ESTADO"]
)

In [0]:
df_meta_br_gold = (
    df_meta_br
    # =====================================================
    # Padronização
    # =====================================================
    .withColumn(
        "REDE",
        initcap(trim(col("REDE")))
    )
    # =====================================================
    # Quantidade de metas disponíveis
    # =====================================================
    .withColumn(
        "QTD_METAS_DEFINIDAS",
        (
            col("META_ALFABETIZACAO_2024").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2025").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2026").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2027").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2028").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2029").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2030").isNotNull().cast("int")
        )

    )
    # =====================================================
    # Possui metas cadastradas
    # =====================================================
    .withColumn(
        "IN_POSSUI_META",
        when(col("QTD_METAS_DEFINIDAS") > 0, 1)
        .otherwise(0)
    )
    # =====================================================
    # Possui taxa observada
    # =====================================================
    .withColumn(
        "IN_POSSUI_RESULTADO",
        when(col("TAXA_ALFABETIZACAO").isNotNull(), 1)
        .otherwise(0)
    )

)

In [0]:
df_meta_br_gold_selected = (
    df_meta_br_gold
    .select(
        "SK_META_ALFABETIZACAO",
        col("ANO").alias('ANO_REFERENCIA'),
        "REDE",
        "TAXA_ALFABETIZACAO",
        "META_ALFABETIZACAO_2024",
        "META_ALFABETIZACAO_2025",
        "META_ALFABETIZACAO_2026",
        "META_ALFABETIZACAO_2027",
        "META_ALFABETIZACAO_2028",
        "META_ALFABETIZACAO_2029",
        "META_ALFABETIZACAO_2030",
        "QTD_METAS_DEFINIDAS",
        "IN_POSSUI_META",
        "IN_POSSUI_RESULTADO",
        "PERCENTUAL_PARTICIPACAO",
        "DT_PROCESSAMENTO",
        "TS_PROCESSAMENTO"

    )

)

In [0]:
validate_primary_key(df_meta_br_gold_selected, "SK_META_ALFABETIZACAO")

validate_not_null(
    df_meta_br_gold_selected,
    [
        "SK_META_ALFABETIZACAO"
    ]
)

validate_years(df_meta_br_gold_selected)

In [0]:
write_delta(
    df=df_meta_br_gold_selected,
    base_path=SILVER_PATH,
    table_name=METAS_BR,
    merge_keys=["SK_META_ALFABETIZACAO"]
)

In [0]:
df_meta_municipio_gold = (
    df_meta_municipio
    # =====================================================
    # Padronização
    # =====================================================
    .withColumn(
        "REDE",
        initcap(trim(col("REDE")))
    )
    # =====================================================
    # Descrição do nível de alfabetização
    # =====================================================
    .withColumn(
        "DS_NIVEL_ALFABETIZACAO",
        when(col("NIVEL_ALFABETIZACAO") == 0, "Abaixo da Meta")
        .when(col("NIVEL_ALFABETIZACAO") == 1, "Meta Atingida")
        .otherwise("Não Informado")
    )
    # =====================================================
    # Quantidade de metas disponíveis
    # =====================================================
    .withColumn(
        "QTD_METAS_DEFINIDAS",
        (
            col("META_ALFABETIZACAO_2024").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2025").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2026").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2027").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2028").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2029").isNotNull().cast("int") +
            col("META_ALFABETIZACAO_2030").isNotNull().cast("int")
        )

    )
    # =====================================================
    # Indicadores
    # =====================================================
    .withColumn(
        "IN_POSSUI_META",
        when(col("QTD_METAS_DEFINIDAS") > 0, 1)
        .otherwise(0)
    )
    .withColumn(
        "IN_POSSUI_RESULTADO",
        when(col("TAXA_ALFABETIZACAO").isNotNull(), 1)
        .otherwise(0)
    )

)

In [0]:
df_meta_municipio_gold_selected = (
    df_meta_municipio_gold
    .select(
        "SK_META_MUNICIPIO",
        col("ANO").alias('ANO_REFERENCIA'),
        "CO_MUNICIPIO",
        "REDE",
        "TAXA_ALFABETIZACAO",
        "META_ALFABETIZACAO_2024",
        "META_ALFABETIZACAO_2025",
        "META_ALFABETIZACAO_2026",
        "META_ALFABETIZACAO_2027",
        "META_ALFABETIZACAO_2028",
        "META_ALFABETIZACAO_2029",
        "META_ALFABETIZACAO_2030",
        "NIVEL_ALFABETIZACAO",
        "DS_NIVEL_ALFABETIZACAO",
        "QTD_METAS_DEFINIDAS",
        "IN_POSSUI_META",
        "IN_POSSUI_RESULTADO",
        "PERCENTUAL_PARTICIPACAO",
        "DT_PROCESSAMENTO",
        "TS_PROCESSAMENTO"
    )
)

In [0]:
validate_primary_key(df_meta_municipio_gold_selected, "SK_META_MUNICIPIO")

validate_not_null(
    df_meta_municipio_gold_selected,
    [
        "SK_META_MUNICIPIO"
    ]
)

validate_years(df_meta_municipio_gold_selected)

In [0]:
write_delta(
    df=df_meta_municipio_gold_selected,
    base_path=SILVER_PATH,
    table_name=METAS_MUNICIPIO,
    merge_keys=["SK_META_MUNICIPIO"]
)